## ML source code map

[sensitivity/runner.py](../../src/ml_classification/sensitivity/runner.py) runs [pipeline.py](../../src/ml_classification/pipeline.py) for each seed.
It uses [shared/config/config.py](../../src/ml_classification/shared/config/config.py) and owns [sensitivity/libraries/](../../src/ml_classification/sensitivity/libraries) for result tables, plots, and HTML reports.
The pipeline executes `step_01_check/` through `step_05_predict/`; each step owns its own `libraries/`.

Source folders use `step_01_check/` through `step_05_predict/`; saved run artifacts still use `01_check/` through `05_predict/`.
See the [source guide](../../src/ml_classification/README.md) and [file usage index](../../src/ml_classification/FILE_USAGE.md) to trace every implementation and its callers.

In [ ]:
from pathlib import Path
import sys
import os
import random
import importlib



from pathlib import Path
from rasterio import features
import numpy as np
import os
import pandas as pd
import random
import sys
import shared.io as io_l
from ml_classification.shared.config.config import ClassificationPipelineConfig
import ml_classification.sensitivity.runner as sensitivity_runner_module

sensitivity_runner_module = importlib.reload(sensitivity_runner_module)
ClassificationSensitivityRunner = sensitivity_runner_module.ClassificationSensitivityRunner

# Global reproducibility seed for notebook-side random operations
random_seed = 42
os.environ["PYTHONHASHSEED"] = str(random_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(random_seed)
np.random.seed(random_seed)

In [ ]:
sample_gdf_file = "E:/Eleni/git_repo/work_dir/test/input_gdf.gpkg"
gdf_sample = io_l.read_data(sample_gdf_file)
gdf_sample = gdf_sample.set_crs("EPSG:2100", allow_override=True)
gdf_sample = gdf_sample[gdf_sample["label"].notna()].reset_index(drop=True)
gdf_train = gdf_sample.copy()

unknown_fraction = 0.20
rng = np.random.default_rng(random_seed)
unknown_count = int(len(gdf_train) * unknown_fraction)
unknown_indices = rng.choice(gdf_train.index.to_numpy(), size=unknown_count, replace=False)
gdf_train.loc[unknown_indices, "label"] = None

features = [col for col in gdf_sample.columns if col not in ["label", "geometry"]]
print(f"Rows: {len(gdf_sample)} | Unknown labels injected: {gdf_train['label'].isna().sum()}")

In [ ]:
# Sensitivity setup
sensitivity_output_dir = Path("E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity")
sensitivity_output_dir.mkdir(parents=True, exist_ok=True)

seed_values = [11, 22, 33, 42]

def build_config_for_seed(seed: int) -> ClassificationPipelineConfig:
    project_dir = sensitivity_output_dir / f"run_seed_{seed}"
    return ClassificationPipelineConfig(
        project_dir=project_dir,
        target_column="label",
        feature_columns=features,
        id_column="row_id",
        cv_folds=3,
        test_size=0.2,
        random_state=seed,
        n_jobs=1,
        max_search_candidates=12,
        selection_type="soft_voting",
        interpretability_top_models=3,
        interpretability_include_shap=False,
        top_voting_models=None,
        # label_balancing_method="random_oversample",
        spatial_split=True,
        spatial_split_grid_size=10,
        selected_models=(
            "gradient_boosting",
            "hist_gradient_boosting",
            "bagging",
            "random_forest",
            "decision_tree",
            "extra_trees",
            "bayesian",
            "xgboost",
            #"lightgbm",
            "knn",
            "neural_network"
    ),
        max_map_geometries=20000,
    )

In [ ]:
runner = ClassificationSensitivityRunner(output_dir=sensitivity_output_dir)

# Option A (all-in-one):
# results_df = runner.run(gdf=gdf_train, seeds=seed_values, config_builder=build_config_for_seed)

# Option B (step 1): run all seeds and save only the CSV summary
results_df = runner.run_seeds(
    gdf=gdf_train,
    seeds=seed_values,
    config_builder=build_config_for_seed,
    write_csv=True,
)

results_df

In [ ]:
# Option B (step 2): create plot + HTML report from an existing results table
results_df = pd.read_csv(sensitivity_output_dir / "sensitivity_results.csv")
artifacts = runner.create_report(results_df, write_csv=False)
artifacts

In [ ]:
import webbrowser
html_path = sensitivity_output_dir / "sensitivity_report.html"
webbrowser.open(html_path.as_uri())